# Notebook 1 — AURKB-focused COCONUT prescreening

**Article:** Chemotype-aware QSAR–GCN consensus screening and molecular modeling prioritize natural-product-derived AURKB-targeting candidates for pancreatic ductal adenocarcinoma

**Target journal:** Molecular Diversity

**Authors:** The author list, affiliations and corresponding-author details are not recorded in the
manuscript file supplied with this repository and are therefore not reproduced here. They are given in
the submitted manuscript.

**Purpose of this notebook:** Standardize the COCONUT natural-product library with RDKit and apply the kinase-oriented, lead-like, PAINS-free prescreening window that defines the compound library entering QSAR screening.

**Workflow stage:** Stage 1 of 3 in the computational screening chain.

**Input:** COCONUT natural-product export (`coconut_csv-05-2026.csv`; 738,827 input records).

**Output:** `Prescreening_coconut_AURKB.csv` — 86,056 standardized, lead-like, PAINS-free compounds, plus chunk-level filter audits, configuration and version logs, and chemical-space figures.

**Relationship to the other stages:** No upstream notebook. The 86,056-compound library is the screening input of Notebook 2. The prescreening workflow itself is implemented in `scripts/aurkb_prescreening_vscode.py`, which this notebook imports.

**Frozen-artifact notice:** The scientific state of this study is frozen. This file is a
presentation-only copy of the executed original notebook. Only Markdown text and Python comments were
edited; executable code, cell order, cell identifiers, execution counts and all stored outputs are
unchanged and were not re-executed for this repository. Historical manuscript-authoritative artifacts
are never replaced by values recalculated in a current software environment.

**Executed outputs:** The outputs stored in this notebook are the outputs of the original recorded
execution and are retained for provenance, including environment/version records, warnings and
software-drift evidence.

---


# Notebook 1 — AURKB-focused COCONUT prescreening

This notebook runs a reproducible prescreening workflow for a large COCONUT/natural-product database before QSAR and GCN modelling of AURKB inhibitors.

The main output for Notebook 2 is:

`<run_dir>/Data/Prescreening_coconut_AURKB.csv`

The workflow is designed for VS Code/Jupyter on a local workstation. Place `aurkb_prescreening_vscode.py` in the same folder as this notebook before running.

## Methodological notes

This stage is a **target-oriented chemical-space enrichment step**, not a definitive AURKB activity predictor. AURKB activity prediction is performed later by the QSAR/GCN models.

The prescreening criteria are intended to:

1. retain kinase/ATP-pocket-compatible, lead-like molecules;
2. retain molecules with sufficient heteroatom and ring features for possible kinase hinge-region interactions;
3. reduce highly planar/aromatic compounds that may behave as non-specific nucleic-acid intercalators;
4. remove PAINS-like structures before ML screening;
5. optionally annotate or filter by similarity to curated AURKB reference inhibitors.

The notebook saves configuration, package versions, filter audit logs, output CSV/Parquet files, and chemical-space figures to support reproducibility in a GitHub repository.

## 1. Environment setup

Install the dependencies in your VS Code Python environment before running:

```bash
pip install -r requirements_aurkb_prescreening.txt
```

If RDKit installation via pip causes issues on Windows, use a conda environment:

```bash
conda create -n aurkb python=3.11 -y
conda activate aurkb
conda install -c conda-forge rdkit pandas numpy tqdm matplotlib pyarrow -y
pip install notebook ipykernel
```

In [6]:
# Optional dependency check
import sys
from pathlib import Path

import pandas as pd
import numpy as np
from rdkit import rdBase

print('Python:', sys.version)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('RDKit:', rdBase.rdkitVersion)

Python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
pandas: 2.2.3
numpy: 2.2.5
RDKit: 2026.03.2


## 2. Configure paths and screening options

Edit `INPUT_FILE` and `OUTPUT_ROOT` before running. Use raw strings (`r"..."`) for Windows paths.

In [7]:
from pathlib import Path

# -------------------------------------------------------------------------
# EDIT THESE PATHS
# -------------------------------------------------------------------------
INPUT_FILE = r"C:\Users\Prottoy\Desktop\PDAC\Plant dataset\Coconut\coconut_csv-05-2026.csv"
OUTPUT_ROOT = r"C:\Users\Prottoy\Desktop\AURKB_Project\Prescreening_Runs"

# Optional: curated AURKB reference ligands with a SMILES column.
# Leave as None unless you have manually curated reference inhibitors.
AURKB_REFERENCE_FILE = None

# -------------------------------------------------------------------------
# RUN OPTIONS
# -------------------------------------------------------------------------
CHUNKSIZE = 100_000       # 100k is reasonable for a 32 GB RAM workstation
APPLY_SUPERCLASS_FILTER = False
REMOVE_PAINS = True
SIMILARITY_THRESHOLD = None  # Example: 0.15 or 0.20 only if using curated AURKB references

print('Input exists:', Path(INPUT_FILE).exists(), INPUT_FILE)
print('Output root:', OUTPUT_ROOT)

Input exists: True C:\Users\Prottoy\Desktop\PDAC\Plant dataset\Coconut\coconut_csv-05-2026.csv
Output root: C:\Users\Prottoy\Desktop\AURKB_Project\Prescreening_Runs


### Important option: `APPLY_SUPERCLASS_FILTER`

The previous notebook restricted the library to selected COCONUT chemical superclasses. For manuscript-level robustness, the default here is `False`, because superclass labels are database annotations and may introduce avoidable selection bias.

Recommended strategy:

- Run first with `APPLY_SUPERCLASS_FILTER = False` for the primary manuscript workflow.
- If the file is still too large, rerun with `APPLY_SUPERCLASS_FILTER = True` and report it clearly as a secondary enrichment step.

Both outputs are reproducible because the configuration is saved automatically.

## 3. Run prescreening

This cell imports the VS Code-compatible workflow script and executes the full pipeline.

In [9]:
# Make sure aurkb_prescreening_vscode.py is in the same directory as this notebook.
from aurkb_prescreening_vscode import PrescreenConfig, run_prescreening

config = PrescreenConfig(
    input_file=INPUT_FILE,
    output_root=OUTPUT_ROOT,
    chunksize=CHUNKSIZE,
    apply_superclass_filter=APPLY_SUPERCLASS_FILTER,
    remove_pains=REMOVE_PAINS,
    aurkb_reference_file=AURKB_REFERENCE_FILE,
    similarity_threshold=SIMILARITY_THRESHOLD,
    write_parquet=True,
    make_figures=True,
)

output_csv = run_prescreening(config)
print('Main output for Notebook 2:')
print(output_csv)

2026-05-20 00:56:31,650 | INFO | Starting AURKB publication-ready prescreening.
2026-05-20 00:56:31,651 | INFO | Input file: C:\Users\Prottoy\Desktop\PDAC\Plant dataset\Coconut\coconut_csv-05-2026.csv
2026-05-20 00:56:31,652 | INFO | Run directory: C:\Users\Prottoy\Desktop\AURKB_Project\Prescreening_Runs\AURKB_prescreen_20260520_005631
2026-05-20 00:56:31,673 | INFO | Detected delimiter: ','


Processing COCONUT chunks: 0chunk [00:00, ?chunk/s]

2026-05-20 03:18:55,687 | INFO | Processed 500,000 rows; cumulative final pass = 58,146
2026-05-20 04:22:29,174 | INFO | Saved final prescreened library: C:\Users\Prottoy\Desktop\AURKB_Project\Prescreening_Runs\AURKB_prescreen_20260520_005631\Data\Prescreening_coconut_AURKB.csv (86,056 compounds)
2026-05-20 04:22:29,179 | WARNING | Could not write Parquet output: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
2026-05-20 04:22:31,156 | INFO | Saved structural-pass library before optional superclass filtering: C:\Users\Prottoy\Desktop\AURKB_Project\Prescreening_Runs\AURKB_prescreen_202605

## 4. Inspect the output library

The output should contain standardized SMILES and RDKit descriptors used for auditability. Notebook 2 should use the `smiles` column from this file.

In [10]:
df = pd.read_csv(output_csv)
print(df.shape)
display(df.head())

essential_cols = ['smiles', 'rdkit_mw', 'rdkit_logp', 'rdkit_tpsa', 'rdkit_qed', 'rdkit_fsp3', 'pains_alert']
missing = [c for c in essential_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

print('Duplicate standardized SMILES:', df['smiles'].duplicated().sum())
print('PAINS alerts retained:', df['pains_alert'].sum() if 'pains_alert' in df.columns else 'not calculated')

(86056, 35)


,identifier,name,smiles,smiles_original,chemical_super_class,rdkit_mw,rdkit_logp,rdkit_tpsa,rdkit_rotatable_bonds,rdkit_qed,...,pass_tpsa,pass_rotatable_bonds,pass_qed,pass_hba,pass_hetero_atoms,pass_ring_count,pass_fsp3,pass_aromatic_rings,pass_pains,pass_aurkb_similarity
0,CNP0329605.2,Heteratisine,CCN1C[C@]2(C)CC[C@H](OC)[C@]34C1[C@H]([C@@H](O...,CCN1C[C@]2(C)CC[C@H](OC)[C@]34C1[C@H]([C@@H](O...,Organoheterocyclic compounds,391.508,1.1853,79.23,2,0.688626,...,True,True,True,True,True,True,True,True,True,True
1,CNP0550908.1,"[(2~{R},5~{S},7~{R},8~{R},9~{S},11~{R},12~{S},...",C=C1[C@@H](O)[C@@]23[C@@H]4C[C@H]5C(CCC[C@]5(C...,C=C1[C@@H](O)[C@@]23[C@@H]4C[C@H]5C(CCC[C@]5(C...,Lipids and lipid-like molecules,406.519,2.4215,85.22,3,0.553116,...,True,True,True,True,True,True,True,True,True,True
2,CNP0360445.0,1282107-63-4,COc1ccc(CCNC(=O)Cn2ncc3ccc(OC)c(OC)c3c2=O)cc1,COC1=CC=C(CCNC(=O)CN2N=CC3=CC=C(OC)C(OC)=C3C2=...,Organoheterocyclic compounds,397.431,1.7812,91.68,8,0.623057,...,True,True,True,True,True,True,True,True,True,True
3,CNP0363386.1,"(3aR)-4-butyl-1,5-dioxo-N-(1,3-thiazol-2-yl)-2...",CCCCN1C(=O)c2ccccc2N2C(=O)CC[C@@]12C(=O)Nc1nccs1,CCCCN1C(=O)C2=CC=CC=C2N2C(=O)CC[C@@]12C(=O)NC1...,Organic acids and derivatives,384.461,2.8608,82.61,5,0.859293,...,True,True,True,True,True,True,True,True,True,True
4,CNP0370288.0,Oprea1_609139,COc1cccc(NC(=O)c2c(O)c3cccc4c3n(c2=O)CC4)c1,COC1=CC=CC(NC(=O)C2=C(O)C3=CC=CC4=C3N(CC4)C2=O...,Benzenoids,336.347,2.5241,80.56,3,0.770068,...,True,True,True,True,True,True,True,True,True,True


Duplicate standardized SMILES: 0
PAINS alerts retained: 0


## 5. Review audit files

The pipeline automatically saves:

- `Data/Prescreening_coconut_AURKB.csv` — final prescreened library for Notebook 2
- `Data/Prescreening_coconut_AURKB.parquet` — optional compressed output
- `Data/Prescreening_coconut_AURKB_structural_all_classes.csv` — structural-pass set before optional superclass filtering
- `Data/Prescreening_chunk_audit.csv` — chunk-level filtering audit
- `Data/Prescreening_aggregate_summary.json` — aggregate run summary
- `Data/Prescreening_chemical_space_summary.csv` — descriptor summary statistics
- `Logs/prescreening_config.json` — exact configuration
- `Logs/software_versions.json` — environment metadata
- `Logs/prescreening.log` — full processing log
- `Figures/*.png` — distribution figures

In [ ]:
run_dir = Path(output_csv).parents[1]
print('Run directory:', run_dir)

print('Data files:')
for p in sorted((run_dir / 'Data').glob('*')):
    print(' -', p.name)

print('Log files:')
for p in sorted((run_dir / 'Logs').glob('*')):
    print(' -', p.name)

print('Figure files:')
for p in sorted((run_dir / 'Figures').glob('*')):
    print(' -', p.name)

Run directory: C:\Users\Prottoy\Desktop\AURKB_Project\Prescreening_Runs\AURKB_prescreen_20260520_005631
Data files:
 - Prescreening_aggregate_summary.json
 - Prescreening_chemical_space_summary.csv
 - Prescreening_chunk_audit.csv
 - Prescreening_coconut_AURKB.csv
 - Prescreening_coconut_AURKB_structural_all_classes.csv
Log files:
 - prescreening.log
 - prescreening_config.json
 - software_versions.json
Figure files:
 - 01_molecular_weight_distribution.png
 - 02_logp_distribution.png
 - 03_tpsa_distribution.png
 - 04_qed_distribution.png
 - 05_fsp3_distribution.png
 - 06_chemical_superclass_distribution.png


: 

## 6. Optional CLI execution from VS Code terminal

The same workflow can be run without Jupyter:

```bash
python aurkb_prescreening_vscode.py   --input "C:/Users/Prottoy/Desktop/PDAC/Plant dataset/Coconut/coconut_csv-05-2026.csv"   --output-root "C:/Users/Prottoy/Desktop/AURKB_Project/Prescreening_Runs"   --chunksize 100000
```

To apply chemical-superclass enrichment:

```bash
python aurkb_prescreening_vscode.py   --input "C:/path/to/coconut_csv-05-2026.csv"   --output-root "C:/path/to/Prescreening_Runs"   --apply-superclass-filter
```

To annotate/filter using curated AURKB reference inhibitors:

```bash
python aurkb_prescreening_vscode.py   --input "C:/path/to/coconut_csv-05-2026.csv"   --output-root "C:/path/to/Prescreening_Runs"   --aurkb-reference-file "C:/path/to/AURKB_reference_inhibitors.csv"   --similarity-threshold 0.15
```

## 7. Methods summary for this stage

The COCONUT natural-product library was processed using a reproducible, chunk-wise cheminformatics workflow. Molecular structures were standardized using RDKit by parsing SMILES strings, retaining the largest molecular fragment, neutralizing charges where possible, and generating canonical isomeric SMILES. Duplicates were removed based on standardized SMILES. RDKit descriptors were calculated for each valid molecule and used to apply a predefined kinase-like prescreening window: molecular weight 250–550 Da, Crippen LogP 1.0–5.0, TPSA 70–140 Å², rotatable bonds ≤10, QED ≥0.50, at least one hydrogen-bond acceptor, at least two heteroatoms, and at least one ring. To reduce enrichment of highly planar non-specific nucleic-acid intercalators, compounds were additionally required to have fraction Csp³ ≥0.15 and ≤4 aromatic rings. PAINS-alert compounds were removed. The resulting standardized library was used as the input for downstream AURKB QSAR screening and applicability-domain filtering.